<a href="https://colab.research.google.com/github/Senadeera-NK/LearnVault/blob/main/LearnVault_QA_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers PyPDF2 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00


In [17]:
from google.colab import files, drive
import os as os

os.makedirs("/temp",exist_ok=True)
uploaded = files.upload() #choose the pdf manually

# move it into tmp
for filename in uploaded.keys():
  os.rename(filename, f"/tmp/{filename}")

Saving IT & Business Professional CV -  S.A. Nanduni Kaveesha Senadeera.docx (1) (1) - Copy.pdf to IT & Business Professional CV -  S.A. Nanduni Kaveesha Senadeera.docx (1) (1) - Copy.pdf


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import PyPDF2
import torch

# Load model correctly
tokenizer = AutoTokenizer.from_pretrained("valhalla/t5-small-qg-hl")
model = AutoModelForSeq2SeqLM.from_pretrained("valhalla/t5-small-qg-hl")


tokenizer_config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

In [22]:
def extract_text_from_pdf(file_path):
    reader = PyPDF2.PdfReader(file_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# ✅ Use a public, open question-generation model
model_name = "iarfmoose/t5-base-question-generator"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_questions(text, qa_type="fast", num_questions=10):
    if qa_type == "mcq":
        prefix = "Generate multiple choice questions: "
    elif qa_type == "truefalse":
        prefix = "Generate true or false questions: "
    else:
        prefix = "Generate short answer questions: "

    prompt = f"{prefix}{text}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_length=512)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [27]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "iarfmoose/t5-base-question-generator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_questions(text):
    prompt = f"generate questions: context: {text}"
    inputs = tokenizer.encode(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(inputs, max_length=256, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# test it
sample_text = """The sun rises in the east and sets in the west.
Earth rotates on its axis from west to east, causing day and night."""
print(generate_questions(sample_text))



where the sun rises and sets in the west


In [28]:
!pip install transformers PyPDF2 torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import PyPDF2, os, torch

# --- Load model ---
model_name = "valhalla/t5-small-qg-hl"  # lightweight and public
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# --- Extract text from PDF ---
def extract_text_from_pdf(file_path):
    text = ""
    reader = PyPDF2.PdfReader(file_path)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# --- Generate questions ---
def generate_questions(text, num_questions=10):
    prompt = f"generate questions: context: {text}"
    inputs = tokenizer.encode(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(inputs, max_length=256, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- Loop through PDFs in /tmp ---
tmp_folder = "/tmp"
for filename in os.listdir(tmp_folder):
    if filename.endswith(".pdf"):
        file_path = os.path.join(tmp_folder, filename)
        print(f"\n📘 Processing: {filename}")
        text = extract_text_from_pdf(file_path)
        if len(text.strip()) == 0:
            print("⚠️ No text found in this PDF.")
            continue
        questions = generate_questions(text)
        print("🧠 Generated Questions:\n", questions)
        print("="*80)



📘 Processing: IT & Business Professional CV -  S.A. Nanduni Kaveesha Senadeera.docx (1) (1) - Copy.pdf
🧠 Generated Questions:
 What is the name of the business and operations support professional?
